# Splitting up observed data into individual conversations

In [1]:
import os
import shutil
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_DIRECTORY = '../data/ckpts/rosen/lme-ready'

In [3]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [4]:
FS = grab_all_files(DATA_DIRECTORY)
print(FS[:3])
len(FS)

['../data/ckpts/rosen/lme-ready/xling-callfriend-49.parquet', '../data/ckpts/rosen/lme-ready/xling-callhome-149.parquet', '../data/ckpts/rosen/lme-ready/xling-callhome-299.parquet']


26

## Process

In [5]:
for obs_f in tqdm(FS):

    df = pd.read_parquet(obs_f)

    grouped = df.groupby(by=['corpus', 'conversation_id'])
    for (corpus, cid), dfi in grouped:
        if 'CANDOR' in obs_f:
            new_file_name = os.path.join(DATA_DIRECTORY, 'CANDOR-' + cid + '.parquet')
        else:
            new_file_name = os.path.join(DATA_DIRECTORY, corpus + '-' + cid + '.parquet')

        if os.path.exists(new_file_name):
            old_d = pd.read_parquet(new_file_name)
            dfi = pd.concat([old_d, dfi], ignore_index=True)

        dfi.to_parquet(
            new_file_name,
            engine='fastparquet',
            compression='snappy'
        )

    os.remove(obs_f)

  0%|          | 0/26 [00:00<?, ?it/s]